# 🧠 Synthetic Instruction Dataset for LLM Fine-Tuning

In this lesson, we are going to generate a synthetic instruction dataset — question-and-answer pairs grounded in real documents, that can be used to fine-tune or evaluate a large language model without exposing the source content.

In this notebook, we will:
1. Generate Q&A pairs from a folder of documents using `DocumentQAGeneration`
2. Inspect and quality-filter the output

## 1. Generate Q&A Pairs

`generate()` accepts a folder path, a single file path, or a PyArrow table as `input_source`. `num_qa_pairs` sets how many pairs to generate **per document**.

In [ ]:
#add here the subcription key
subscription_key='add-your-subscription-key' 

In [9]:
from ydata.synthesizers.text.model.qa import DocumentQAGeneration

qa_generator = DocumentQAGeneration(
    backend='openai',
    subscription_key=subscription_key,
    model='gpt-5'
)

print('DocumentQAGeneration ready ✓')

DocumentQAGeneration ready ✓


In [10]:
# Generate Q&A pairs from a folder of documents
# Replace 'assets/documents/' with the path to your own document folder
result = qa_generator.generate(
    input_source='../data/documents/',   # folder, single file, or PyArrow table
    docs_extension='docx',              # 'docx' | 'txt' | 'pdf'
    num_qa_pairs=5,                     # pairs per document
)

print(f'Generated {len(result)} Q&A pairs ✓')

100%|██████████| 2/2 [00:00<00:00,  3.36it/s]


INFO: 2026-08-27 16:32:32,652 HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO: 2026-08-27 16:32:35,182 HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
Generated 85 Q&A pairs ✓


In [11]:
print(f'Shape: {result.shape}')
print(f'Columns: {list(result.columns)}')
result

Shape: (85, 4)
Columns: ['question', 'answer', 'difficulty', 'source']


,question,answer,difficulty,source
0,What is the title of the document?,Exploratory Data Analysis Is a Data Engineerin...,easy,../data/documents/Technical blog article_0b0b7...
1,What is the estimated reading time stated in t...,14 min read,easy,../data/documents/Technical blog article_0b0b7...
2,What category is the document listed under?,Data Engineering,easy,../data/documents/Technical blog article_0b0b7...
3,Name five tags mentioned in the document.,"EDA, Data Quality, Data Observability, Spark, ...",easy,../data/documents/Technical blog article_0b0b7...
4,List all seven aspects of EDA as an operationa...,"Structural analysis, Distributional analysis, ...",medium,../data/documents/Technical blog article_0b0b7...
...,...,...,...,...
80,Which configuration or design heuristics are s...,Use broadcast joins for small dimensions under...,hard,../data/documents/Technical blog article_f02c7...
81,What time zone and timestamp hygiene practices...,Check distributions of time zones and DST tran...,medium,../data/documents/Technical blog article_f02c7...
82,Which audiences and stacks are explicitly call...,Data engineers in North America working across...,easy,../data/documents/Technical blog article_f02c7...
83,What is the call to action given at the end of...,"Pick your most critical pipeline, run a focuse...",easy,../data/documents/Technical blog article_f02c7...


In [12]:
# Distribution of Q&A pairs across source documents
print('Pairs per source document:')
print(result['source'].value_counts().to_string())

Pairs per source document:
source
../data/documents/Technical blog article_f02c74c1-65b9-4deb-bd3c-c447689e896d.docx    34
../data/documents/Technical blog article_0b0b701d-100d-4852-a55e-8c8f3856ad7b.docx    30


In [13]:
# Spot-check a single Q&A pair
sample = result.sample(1).iloc[0]
print(f'Source  : {sample["source"]}')
print(f'Question: {sample["question"]}')
print(f'Answer  : {sample["answer"]}')

Source  : ../data/documents/Technical blog article_0b0b701d-100d-4852-a55e-8c8f3856ad7b.docx
Question: How should overlap between fact and dimension tables be evaluated?
Answer  : Measure overlap of join keys and report orphan rates and expected fan-out.


## 2. Export to JSONL for Fine-Tuning

Instruction fine-tuning pipelines expect data in JSONL format — one JSON object per line. `question` maps to `instruction` and `answer` maps to `response`.

In [14]:
import os
import json 
output_path = 'output/instruction_dataset.jsonl'
os.makedirs('output', exist_ok=True)

with open(output_path, 'w') as f:
    for _, row in result.iterrows():
        record = {
            'instruction': row['question'],
            'response':    row['answer'],
            'source':      row['source'],
        }
        f.write(json.dumps(record) + '\n')

print(f'Exported {len(result)} pairs → {output_path}')

Exported 85 pairs → output/instruction_dataset.jsonl


In [ ]:
# Preview the first three records
with open(output_path) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        print(json.dumps(json.loads(line), indent=2))
        print()

## ✅ Summary

In this lesson you:

1. **Configured an LLM backend** — wired an Anthropic or OpenAI key into the environment so `DocumentQAGeneration` can call it internally
2. **Generated Q&A pairs** using `DocumentQAGeneration.generate()`, pointed at a folder of documents with a specified file type and pairs-per-document count
3. **Inspected the output** — a DataFrame with `question`, `answer`, and `source` columns, traceable back to each source file

The key insight: synthetic instruction datasets are only as good as their source documents. Combining `DocumentGenerator` (from the previous video) with `DocumentQAGeneration` gives you a fully synthetic pipeline — generate the documents, then generate the instruction dataset from them.